# Mapping with Predefined Lists

In [301]:
import os
import findspark
findspark.init()
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
import List as mapping_list

In [302]:
import importlib
importlib.reload(mapping_list)

<module 'List' from '/mnt/01DBA8B8279979A0/Pulse - E-Commerce Data Analytics Engine/mapping/List.py'>

In [303]:
def normalize_dataframe(df, column_variants):
    variant_to_standard = {
        v.lower(): std_col
        for std_col, variants in column_variants.items()
        for v in variants
    }

    mapped_cols = {}
    new_columns = []
    for col in df.columns:
        col_lower = col.lower()
        if col_lower in variant_to_standard:
            std_col = variant_to_standard[col_lower]
            new_columns.append(std_col)
            mapped_cols[std_col] = col
        else:
            new_columns.append(col)

    for old_col, new_col in zip(df.columns, new_columns):
        df = df.withColumnRenamed(old_col, new_col)

    missing_cols = []
    for std_col in column_variants.keys():
        if std_col not in df.columns:
            df = df.withColumn(std_col, lit(None))
            missing_cols.append(std_col)

    schema_cols = list(column_variants.keys())
    extra_cols = [c for c in df.columns if c not in schema_cols]

    new_df = df.select(schema_cols)
    df_extra = df.select(schema_cols + extra_cols)

    return new_df, df_extra, extra_cols, missing_cols, mapped_cols

In [304]:
base_dir = os.getcwd()
file_path_excel = os.path.join(base_dir, "./../faker/messy_inventory_data.xlsx")
file_path_csv = os.path.join(base_dir, "./../faker/messy_inventory_data.csv")

In [305]:
excel_df = pd.read_excel(file_path_excel, engine="openpyxl")
excel_df.to_csv(file_path_csv, index=False)

In [306]:
spark = SparkSession.builder.appName("NormalizeData").getOrCreate()
df = spark.read.csv(file_path_csv, header=True, inferSchema=True)

In [307]:
df.show(5)

+------+-----------+------------+-------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|inv_id|product_ref|   vendor_id|current_stock|reserved_stock|min_stock_level|   last_restock_date|      last_sale_date|monthly_storage_cost|        created_date|available_qty|days_since_last_sale|stock_status|warehouse_location|total_stock_value|restock_lead_time_days|expiry_date|
+------+-----------+------------+-------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|100555|  PROD_0781|    SUPP_017|           72|            21|             69|2025-08-25 18:33:...|2025-09-07 03:54:...|                0.74|2024-05-31

In [308]:
new_df, extra_df, extra_cols, missing, mapped = normalize_dataframe(
    df, mapping_list.mapping_dict_inventory
)

In [309]:
print("\nNormalized DataFrame:")
new_df.show(5)

print("\nDataFrame with Extra Columns:")
extra_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)


Normalized DataFrame:
+------------+----------+------------+--------------+-----------------+-------------+-------------------+--------------+------------+----------+
|inventory_id|product_id| supplier_id|stock_quantity|reserved_quantity|reorder_level|last_restocked_date|last_sold_date|storage_cost|created_at|
+------------+----------+------------+--------------+-----------------+-------------+-------------------+--------------+------------+----------+
|      100555| PROD_0781|    SUPP_017|            72|             NULL|         NULL|               NULL|          NULL|        NULL|      NULL|
|      100938| PROD_0373|    SUPP_003|             8|             NULL|         NULL|               NULL|          NULL|        NULL|      NULL|
|      100060| PROD_9999|  SUPP_029  |        Many  |             NULL|         NULL|               NULL|          NULL|        NULL|      NULL|
|      100693| PROD_0244|    SUPP_022|         240  |             NULL|         NULL|               NULL|  

# Implementing NLTK

In [310]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.metrics.distance import edit_distance
from difflib import SequenceMatcher

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     /home/khalid_ah_1/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/khalid_ah_1/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/khalid_ah_1/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/khalid_ah_1/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/khalid_ah_1/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

#### Implementing Jaccard Similarity

In [311]:
def jaccard_similarity(source_column, target_column):
    source_col = source_column.lower()
    target_col = target_column.lower()
    intersection = len(set(source_col).intersection(set(target_col)))
    union = len(set(source_col).union(set(target_col)))
    jaccard_similarity = intersection / union if union > 0 else 0
    return jaccard_similarity

#### Implementing Sequence Matcher

In [312]:
def sequence_matching(source_column, target_column):
    source_col = source_column.lower()
    target_col = target_column.lower()
    return SequenceMatcher(None, source_col, target_col).ratio()

#### Implementing Edit Distance

In [313]:
def editing_distance(source_column, target_column):
    max_len = max(len(source_column), len(target_column))
    return 1 - (edit_distance(source_col.lower(), target_col.lower()) / max_len)

#### Combining them all

In [314]:
def mapping_with_combination(df, missing_cols, extra_cols, mapped_cols, threshold=0.87):
    for missing_col in missing_cols[:]:
            for extra_col in extra_cols[:]:
                final = 0.4 * jaccard_similarity(missing_col, extra_col) + 0.3 * sequence_matching(missing_col, extra_col) + 0.3 * edit_distance(missing_col, extra_col)
                if final >= threshold:
                    print(f"Mapping: {extra_col} -> {missing_col}")
                    mapped_cols[missing_col] = extra_col
                    missing_cols.remove(missing_col)
                    extra_cols.remove(extra_col)
                    break
    for new_col, old_col in mapped_cols.items():
        df = df.withColumnRenamed(old_col, new_col)

    return df, missing_cols, extra_cols, mapped_cols


new_df, missing, extra_cols, mapped = mapping_with_combination(
    df, missing, extra_cols, mapped, threshold=0.87
)

print("\nNormalized DataFrame:")
new_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)

Mapping: reserved_stock -> reserved_quantity
Mapping: min_stock_level -> reorder_level
Mapping: last_restock_date -> last_restocked_date
Mapping: last_sale_date -> last_sold_date
Mapping: monthly_storage_cost -> storage_cost
Mapping: created_date -> created_at

Normalized DataFrame:
+------------+----------+------------+--------------+-----------------+-------------+--------------------+--------------------+------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|inventory_id|product_id| supplier_id|stock_quantity|reserved_quantity|reorder_level| last_restocked_date|      last_sold_date|storage_cost|          created_at|available_qty|days_since_last_sale|stock_status|warehouse_location|total_stock_value|restock_lead_time_days|expiry_date|
+------------+----------+------------+--------------+-----------------+-------------+--------------------+--------------------+------------+------------